# 高速特定投資株式情報抽出器\n\n解凍済みXBRLファイルから特定投資株式情報を並行処理で高速抽出します。\n\n## 業務要件\n- 全上場企業の特定投資株式保有情報を抽出\n- 提出会社情報と保有証券情報を結合\n- CSVファイルで出力

## 1. 必要なライブラリのインポート

In [1]:
import os\nimport re\nimport pandas as pd\nfrom bs4 import BeautifulSoup\nimport warnings\nfrom datetime import datetime\nfrom concurrent.futures import ProcessPoolExecutor, as_completed\nimport subprocess\nfrom pathlib import Path\nimport multiprocessing as mp\n\nwarnings.filterwarnings('ignore')\n\n# CPU数の取得\nCPU_COUNT = mp.cpu_count()\nprint(f\"利用可能CPU数: {CPU_COUNT}\")\nprint(\"ライブラリインポート完了\")

SyntaxError: unexpected character after line continuation character (3921608422.py, line 1)

## 2. 高速処理用関数の定義

In [2]:
def find_xbrl_files():\n    \"\"\"grepを使用して特定投資株式情報を含むファイルを高速検索\"\"\"\n    xbrl_path = \"/Users/satoki252595/work/0000_kabulab/0001_MarketableSecuritiesAnalysis/xbrl\"\n    \n    # 0104010_honbun ファイルを検索\n    cmd = f\"find {xbrl_path} -name '*0104010_honbun*.htm' -type f\"\n    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)\n    \n    if result.returncode != 0:\n        print(f\"エラー: {result.stderr}\")\n        return []\n    \n    files = result.stdout.strip().split('\\n')\n    files = [f for f in files if f.strip()]\n    \n    print(f\"対象ファイル数: {len(files)}\")\n    return files\n\ndef extract_company_info_from_filename(filepath):\n    \"\"\"ファイルパスから会社情報を抽出\"\"\"\n    filename = os.path.basename(filepath)\n    \n    # 例: 0104010_honbun_jpcrp030000-asr-001_E02840-000_2024-03-31_01_2024-06-28_ixbrl.htm\n    pattern = r'_([E]\\d+)-(\\d+)_(\\d{4}-\\d{2}-\\d{2})_'\n    match = re.search(pattern, filename)\n    \n    if match:\n        return {\n            'filing_company_code': match.group(1),\n            'document_id': match.group(2),\n            'filing_date': match.group(3),\n            'file_path': filepath\n        }\n    return None\n\nprint(\"基本関数定義完了\")

SyntaxError: unexpected character after line continuation character (2599178888.py, line 1)

In [3]:
def extract_securities_from_file(filepath):\n    \"\"\"単一ファイルから特定投資株式情報を抽出\"\"\"\n    try:\n        # ファイルから会社情報を抽出\n        company_info = extract_company_info_from_filename(filepath)\n        if not company_info:\n            return []\n        \n        # ファイルを読み込み\n        with open(filepath, 'r', encoding='utf-8') as f:\n            content = f.read()\n        \n        # 特定投資株式情報が含まれているかチェック\n        if 'SpecifiedInvestment' not in content:\n            return []\n        \n        soup = BeautifulSoup(content, 'lxml-xml')\n        results = []\n        \n        # 特定投資株式名を検索\n        name_patterns = [\n            r'NameOfSecuritiesDetailsOfSpecifiedInvestment',\n            r'NameOfIssuerDetailsOfSpecifiedInvestment',\n            r'NameOfSecurities.*SpecifiedInvestment',\n            r'NameOfIssuer.*SpecifiedInvestment'\n        ]\n        \n        name_elements = []\n        for pattern in name_patterns:\n            elements = soup.find_all('ix:nonNumeric', {'name': re.compile(pattern)})\n            name_elements.extend(elements)\n        \n        # 重複を除去\n        unique_contexts = {}\n        for elem in name_elements:\n            context = elem.get('contextRef', '')\n            if context and ('CurrentYear' in context or 'Current' in context or 'Instant' in context):\n                if context not in unique_contexts:\n                    unique_contexts[context] = elem\n        \n        # 株式数と貸借対照表計上額を検索\n        shares_elements = soup.find_all('ix:nonFraction', {'name': re.compile(r'NumberOfSharesHeld.*SpecifiedInvestment')})\n        book_value_elements = soup.find_all('ix:nonFraction', {'name': re.compile(r'BookValue.*SpecifiedInvestment')})\n        purpose_elements = soup.find_all('ix:nonNumeric', {'name': re.compile(r'PurposeOfShareholding.*SpecifiedInvestment')})\n        \n        # データを結合\n        for context, name_elem in unique_contexts.items():\n            security_name = name_elem.get_text(strip=True)\n            if not security_name:\n                continue\n            \n            # 対応する株式数を検索\n            shares = None\n            for shares_elem in shares_elements:\n                if shares_elem.get('contextRef') == context:\n                    shares = shares_elem.get_text(strip=True).replace(',', '')\n                    break\n            \n            # 対応する簿価を検索\n            book_value = None\n            for book_elem in book_value_elements:\n                if book_elem.get('contextRef') == context:\n                    book_value = book_elem.get_text(strip=True).replace(',', '')\n                    break\n            \n            # 保有目的を検索\n            purpose = None\n            for purpose_elem in purpose_elements:\n                if purpose_elem.get('contextRef') == context:\n                    purpose = purpose_elem.get_text(strip=True)\n                    break\n            \n            # データが存在する場合のみ追加\n            if shares or book_value or purpose:\n                # 証券コードを抽出\n                stock_code_match = re.search(r'(\\d{4})', security_name)\n                stock_code = stock_code_match.group(1) if stock_code_match else None\n                \n                # 数値の安全な変換\n                try:\n                    shares_int = int(shares) if shares and shares.isdigit() else None\n                except:\n                    shares_int = None\n                \n                try:\n                    book_value_float = float(book_value) if book_value and book_value.replace('.', '').isdigit() else None\n                except:\n                    book_value_float = None\n                \n                result = {\n                    'filing_company_code': company_info['filing_company_code'],\n                    'filing_company_name': None,  # 簡素化のため省略\n                    'filing_stock_code': None,    # 簡素化のため省略\n                    'filing_date': company_info['filing_date'],\n                    'document_id': company_info['document_id'],\n                    'held_security_name': security_name,\n                    'held_stock_code': stock_code,\n                    'held_shares': shares_int,\n                    'book_value_million_yen': book_value_float,\n                    'holding_purpose': purpose if purpose else 'N/A'\n                }\n                results.append(result)\n        \n        return results\n        \n    except Exception as e:\n        print(f\"エラー処理中 {filepath}: {str(e)}\")\n        return []\n\nprint(\"データ抽出関数定義完了\")

SyntaxError: unexpected character after line continuation character (3059973569.py, line 1)

## 3. 並行処理での高速抽出実行

In [ ]:
def process_files_parallel(files, max_workers=None):\n    \"\"\"並行処理でファイルを処理\"\"\"\n    if max_workers is None:\n        max_workers = min(CPU_COUNT, len(files))\n    \n    print(f\"並行処理開始: {max_workers} workers\")\n    \n    all_results = []\n    processed = 0\n    \n    with ProcessPoolExecutor(max_workers=max_workers) as executor:\n        # 全ファイルを並行処理にサブミット\n        future_to_file = {executor.submit(extract_securities_from_file, file): file for file in files}\n        \n        # 完了したタスクから結果を取得\n        for future in as_completed(future_to_file):\n            file = future_to_file[future]\n            try:\n                result = future.result()\n                all_results.extend(result)\n                processed += 1\n                \n                # 進捗表示\n                if processed % 500 == 0:\n                    print(f\"処理済み: {processed}/{len(files)} ファイル, 抽出レコード数: {len(all_results)}\")\n                    \n            except Exception as e:\n                print(f\"処理エラー {file}: {str(e)}\")\n    \n    print(f\"並行処理完了: {processed} ファイル処理, {len(all_results)} レコード抽出\")\n    return all_results\n\n# 開始時刻を記録\nstart_time = datetime.now()\nprint(f\"処理開始: {start_time}\")\n\n# 対象ファイル検索\ntarget_files = find_xbrl_files()\n\nif not target_files:\n    print(\"対象ファイルが見つかりません\")\nelse:\n    # 並行処理で抽出実行\n    results = process_files_parallel(target_files)\n    \n    # 終了時刻を記録\n    end_time = datetime.now()\n    processing_time = end_time - start_time\n    \n    print(f\"\\n=== 処理完了 ===\")\n    print(f\"処理時間: {processing_time}\")\n    print(f\"抽出レコード数: {len(results)}\")\n    \n    if results:\n        unique_companies = len(set(r['filing_company_code'] for r in results))\n        print(f\"提出会社数: {unique_companies}\")

## 4. 結果の保存

In [ ]:
if results:\n    # DataFrameに変換\n    df = pd.DataFrame(results)\n    \n    # CSVファイルに保存\n    output_filename = f'fast_marketable_securities_analysis_{datetime.now().strftime(\"%Y%m%d_%H%M%S\")}.csv'\n    df.to_csv(output_filename, index=False, encoding='utf-8-sig')\n    \n    print(f\"\\n=== 保存完了 ===\")\n    print(f\"ファイル名: {output_filename}\")\n    print(f\"レコード数: {len(df)}\")\n    print(f\"提出会社数: {df['filing_company_code'].nunique()}\")\n    \n    # 基本統計\n    print(f\"\\n=== 基本統計 ===\")\n    print(f\"保有証券数: {df['held_security_name'].nunique()}\")\n    \n    # 保有金額の統計\n    book_values = df['book_value_million_yen'].dropna()\n    if not book_values.empty:\n        print(f\"総保有金額: {book_values.sum():,.0f} 百万円\")\n        print(f\"平均保有金額: {book_values.mean():,.0f} 百万円\")\n    \n    # 上位保有会社\n    top_companies = df.groupby('filing_company_code').size().sort_values(ascending=False).head(10)\n    print(f\"\\n=== 上位保有会社 ===\")\n    print(top_companies)\n    \n    # データサンプル表示\n    print(f\"\\n=== データサンプル ===\")\n    print(df[['filing_company_code', 'held_security_name', 'book_value_million_yen']].head(10))\n    \nelse:\n    print(\"抽出されたデータがありません\")

## 5. 処理結果サマリー

In [ ]:
print(f\"\\n=== 最終処理結果 ===\")\nprint(f\"開始時刻: {start_time}\")\nprint(f\"終了時刻: {end_time}\")\nprint(f\"処理時間: {processing_time}\")\nprint(f\"処理ファイル数: {len(target_files)}\")\nprint(f\"抽出レコード数: {len(results) if results else 0}\")\nprint(f\"CPU使用数: {CPU_COUNT}\")\nprint(f\"出力ファイル: {output_filename if results else 'なし'}\")\nprint(\"\\n処理完了！\")